In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from tobas_std_tools_py.control.simulator import LinearSimulator, NonlinearSimulator
from tobas_std_tools_py.control.kalman_filter import LinearKalmanFilter, ExtendedKalmanFilter, UnscentedKalmanFilter
from tobas_std_tools_py.control.c2d import C2D_RK4

In [ ]:
# ウィナー過程(カルマンフィルタ入門, p.116)

A = np.array([[1.]])
Bu = np.zeros((1, 0))
Bv = np.array([[1.]])
C = np.array([[1.]])
Q = np.array([[1.]])  # システム雑音
# R = np.array([[2.]])  # 観測雑音
R = np.array([[10.]])  # 観測雑音

sim_steps = 300

plant = LinearSimulator(A, Bu, Bv, C, Q, R)
kf = LinearKalmanFilter(A, Bu, Bv, C, Q, R)

u = np.zeros((0,))  # 制御入力はなし
steps = np.arange(0, sim_steps)
y_buf = np.empty((sim_steps,))
x_pred_buf = np.empty((sim_steps,))
x_true_buf = np.empty((sim_steps,))

for k in tqdm(steps):
    y = plant.step(u)
    x_true = plant.x_true
    x_pred = kf.step(y, u)

    y_buf[k] = y
    x_true_buf[k] = x_true
    x_pred_buf[k] = x_pred

plt.figure(figsize=(12, 9))
plt.plot(steps, y_buf, label='y')
plt.plot(steps, x_true_buf, label='x_true')
plt.plot(steps, x_pred_buf, label='x_pred')
plt.xlabel(r'$k$')
plt.legend()
plt.show()


In [ ]:
# カルマンフィルタ入門, p.150, 演習問題6-3

alpha = 1.
sim_steps = 300
x_dim = 2

A = np.array([
    [1., 1.],
    [0., 1.],
])
Bu = np.zeros((2, 0))
Bv = np.array([
    [0.],
    [1.],
])
C = np.array([
    [1., 0.],
])
Q = np.array([[10.]])  # システム雑音
R = np.array([[20.]])  # 観測雑音

init_P = np.diag([alpha, alpha])

plant = LinearSimulator(A, Bu, Bv, C, Q, R)
kf = LinearKalmanFilter(A, Bu, Bv, C, Q, R, init_P=init_P)

u = np.zeros((0,))  # 制御入力はなし
steps = np.arange(0, sim_steps)
y_buf = np.empty((sim_steps,))
x_pred_buf = np.empty((sim_steps, x_dim))
x_true_buf = np.empty((sim_steps, x_dim))

for k in tqdm(steps):
    y = plant.step(u)
    x_true = plant.x_true
    x_pred = kf.step(y, u)
    y_buf[k] = y
    x_true_buf[k, :] = x_true
    x_pred_buf[k, :] = x_pred

plt.figure(figsize=(12, 9))
plt.plot(steps, y_buf, label='y')
plt.xlabel(r'$k$')
plt.legend()

for i in range(x_dim):
    plt.figure(figsize=(12, 9))
    plt.plot(steps, x_true_buf[:, i], label=f'x_true_{i}')
    plt.plot(steps, x_pred_buf[:, i], label=f'x_pred_{i}')
    plt.xlabel(r'$k$')
    plt.legend()

plt.show()


In [ ]:
# カルマンフィルタ入門, p.150, 演習問題6-3

sim_steps = 100
x_dim = 3

A = np.array([
    [1.1269, -0.4940, 0.1129],
    [1., 0., 0.],
    [0., 1., 0.],
])
Bu = Bv = np.array([
    [-0.3832],
    [0.5919],
    [0.5191],
])
C = np.array([
    [1., 0., 0.],
])
Q = np.array([[1.]])  # システム雑音
R = np.array([[1.]])  # 観測雑音

plant = LinearSimulator(A, Bu, Bv, C, Q, R)
kf = LinearKalmanFilter(A, Bu, Bv, C, Q, R)

steps = np.arange(0, sim_steps)
y_buf = np.empty((sim_steps,))
x_pred_buf = np.empty((sim_steps, x_dim))
x_true_buf = np.empty((sim_steps, x_dim))

for k in tqdm(steps):
    u = np.array([np.sin(k / 5.)])
    y = plant.step(u)
    x_true = plant.x_true
    x_pred = kf.step(y, u)
    y_buf[k] = y
    x_true_buf[k, :] = x_true
    x_pred_buf[k, :] = x_pred

plt.figure(figsize=(12, 9))
plt.plot(steps, y_buf, label='y')
plt.xlabel(r'$k$')
plt.legend()

for i in range(x_dim):
    plt.figure(figsize=(12, 9))
    plt.plot(steps, x_true_buf[:, i], label=f'x_true_{i}')
    plt.plot(steps, x_pred_buf[:, i], label=f'x_pred_{i}')
    plt.xlabel(r'$k$')
    plt.legend()

plt.show()


In [ ]:
# カルマンフィルタ入門, p.159, 例題7.1

sim_steps = 100
x_dim = 1
y_dim = 1
u_dim = 0


def fx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return x + 3. * np.cos(x / 10.)


def fu(u: np.ndarray) -> np.ndarray:
    assert u.shape == (u_dim,)
    return np.zeros((x_dim,))


def h(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return x**3


def dfx_dx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return 1. - 0.3 * np.sin(x / 10.)


def dh_dx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return 3. * x**2


Bv = np.array([[1.]])
Q = np.array([[1.]])  # システム雑音
R = np.array([[100.]])  # 観測雑音
init_x = np.array([10.])

plant = NonlinearSimulator(x_dim, u_dim, y_dim, fx, fu, h, Bv, Q, R, init_x)
kf = ExtendedKalmanFilter(x_dim, u_dim, y_dim, fx, fu, h, dfx_dx, dh_dx, Bv, Q, R, init_x)

u = np.zeros((u_dim,))  # 制御入力はなし
steps = np.arange(0, sim_steps)
x_pred_buf = np.empty((sim_steps,))
x_true_buf = np.empty((sim_steps,))

for k in tqdm(steps):
    y = plant.step(u)
    x_true = plant.x_true
    x_pred = kf.step(y, u)
    x_true_buf[k] = x_true
    x_pred_buf[k] = x_pred

plt.figure(figsize=(12, 9))
plt.plot(steps, x_true_buf, label='x_true')
plt.plot(steps, x_pred_buf, label='x_pred')
plt.xlabel(r'$k$')
plt.legend()

plt.show()


In [ ]:
# カルマンフィルタ入門, p.174, 例題7.3
# FIXME: Pが発散してうまく機能しないため一旦コメントアウトしている(2022/11/26)

# ダイナミクス
x_dim = 3
y_dim = 1
u_dim = 0

rho_0 = 1.23
eta = 6e+3
g = 9.81
M = 3e+4
a = 3e+4

# UKFの設定
kappa = 0.

# シミュレーションの設定
sim_time = 30.
T = 0.5  # 離散化周期
sim_steps = int(sim_time / T)


def fx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)

    res = np.empty((3,))
    res[0] = x[0] + T * x[1]
    res[1] = x[1] + T * (0.5 * rho_0 * np.exp(-x[0] / eta) * x[1]**2 * x[2] - g)
    res[2] = x[2]

    return res


def fu(u: np.ndarray) -> np.ndarray:
    assert u.shape == (u_dim,)
    return np.zeros((x_dim,))


def h(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return np.sqrt(M**2 + (x[0] - a)**2)


# Bv = np.identity(x_dim)
# Q = np.zeros((x_dim, x_dim))  # システム雑音はなし
# R = np.array([[4e+3]])  # 観測雑音
# init_x = np.array([9e+4, -6e+3, 3e-3])
# init_P = np.diag([9e+3, 4e+5, 0.4])

# plant = NonlinearSimulator(x_dim, u_dim, y_dim, fx, fu, h, Bv, Q, R, init_x)
# kf = UnscentedKalmanFilter(x_dim, u_dim, y_dim, fx, fu, h, Bv, Q, R, kappa, init_x, init_P)

# u = np.zeros((u_dim,))  # 制御入力はなし
# steps = np.arange(0, sim_steps)
# x_pred_buf = np.empty((sim_steps, x_dim))
# x_true_buf = np.empty((sim_steps, x_dim))

# for k in tqdm(steps):
#     y = plant.step(u)
#     x_true = plant.x_true
#     x_pred = kf.step(y, u)
#     x_true_buf[k, :] = x_true
#     x_pred_buf[k, :] = x_pred

# fig = plt.figure(figsize=(12, 9 * x_dim))
# for i in range(x_dim):
#     ax = fig.add_subplot(int(f'{x_dim}1{i + 1}'))
#     ax.plot(steps, x_true_buf[:, i], label=f'x_true_{i + 1}')
#     ax.plot(steps, x_pred_buf[:, i], label=f'x_pred_{i + 1}')
#     ax.legend()

# plt.show()


In [ ]:
# カルマンフィルタ入門, p.187, 例題7.5
# FIXME: 正しく推定できない(2022/11/28)

# 拡張ダイナミクス
x_dim = 4
y_dim = 1
u_dim = 0

M = 2.
C = 1.
K = 0.7

# UKFの設定
kappa = 0.

# シミュレーションの設定
sim_time = 30.
T = 0.01  # 離散化周期
sim_steps = int(sim_time / T)


def fx_cont_plant(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)

    x1 = x[0]
    x2 = x[1]
    u = x[2]
    c = C  # プラントではCが既知

    xd = np.array([
        x2,
        -(K/M)*x1 - (c/M)*x2 + (1/M)*u,
        0.,
        0.,
    ])
    return xd


fx_plant = C2D_RK4(fx_cont_plant, T)


def fx_cont_model(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)

    x1 = x[0]
    x2 = x[1]
    u = x[2]
    c = x[3]  # モデルではCが未知

    xd = np.array([
        x2,
        -(K/M)*x1 - (c/M)*x2 + (1/M)*u,
        0.,
        0.,
    ])
    return xd


fx_model = C2D_RK4(fx_cont_model, T)


def fu(u: np.ndarray) -> np.ndarray:
    assert u.shape == (u_dim,)
    return np.zeros((x_dim,))


def h(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return x[0]


Bv = np.identity(x_dim)
Q = np.identity(x_dim) * 1e-5  # システム雑音
R = np.array([[0.1]])  # 観測雑音
init_x = np.array([0., 0., 0., C])
init_P = np.identity(x_dim)

plant = NonlinearSimulator(x_dim, u_dim, y_dim, fx_plant, fu, h, Bv, Q, R, init_x)
kf = UnscentedKalmanFilter(x_dim, u_dim, y_dim, fx_model, fu, h, Bv, Q, R, kappa, init_x, init_P)

dummy_u = np.zeros((u_dim,))
steps = np.arange(0, sim_steps)
x_pred_buf = np.empty((sim_steps, x_dim))
x_true_buf = np.empty((sim_steps, x_dim))

for k in tqdm(steps):
    # かなりキモい実装だが制御入力はプラントとKFの状態を直接いじることで実現
    t = k * T
    u = 10. * np.sin(t)
    plant._x[2] = u
    kf._x_post[2] = u

    y = plant.step(dummy_u)
    x_true = plant.x_true
    x_pred = kf.step(y, dummy_u)
    x_true_buf[k, :] = x_true
    x_pred_buf[k, :] = x_pred

labels = ['x1', 'x2', 'u', 'C']
fig = plt.figure(figsize=(12, 9 * x_dim))
for i in range(x_dim):
    ax = fig.add_subplot(int(f'{x_dim}1{i + 1}'))
    ax.plot(steps, x_true_buf[:, i], label=labels[i] + '_true')
    ax.plot(steps, x_pred_buf[:, i], label=labels[i] + '_pred')
    ax.legend()

plt.show()


In [ ]:
# カルマンフィルタ入門, p.190, 演習問題7-2

# ダイナミクス
x_dim = 1
y_dim = 1
u_dim = 0

# UKFの設定
kappa = 0.

# シミュレーションの設定
sim_steps = 100


def fx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    x_next = -0.1 * x + np.sin(x)
    return x_next


def fu(u: np.ndarray) -> np.ndarray:
    assert u.shape == (u_dim,)
    return np.zeros((x_dim,))


def h(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    y = x**2
    return y


def dfx_dx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return -0.1 + np.cos(x)


def dh_dx(x: np.ndarray) -> np.ndarray:
    assert x.shape == (x_dim,)
    return 2. * x


Bv = np.identity(x_dim)
Q = np.diag([1.])  # システム雑音
R = np.diag([0.5])  # 観測雑音
init_x = np.array([0.])  # 初期値が0の場合，EKFだと内部ダイナミクスの状態は0から変化しないため推定できない
init_P = np.diag([1.])

plant = NonlinearSimulator(x_dim, u_dim, y_dim, fx, fu, h, Bv, Q, R, init_x)
ekf = ExtendedKalmanFilter(x_dim, u_dim, y_dim, fx, fu, h, dfx_dx, dh_dx, Bv, Q, R, init_x, init_P)
ukf = UnscentedKalmanFilter(x_dim, u_dim, y_dim, fx, fu, h, Bv, Q, R, kappa, init_x, init_P)

u = np.zeros((u_dim,))  # 制御入力はなし
steps = np.arange(0, sim_steps)
true_xs = np.empty((sim_steps, x_dim))
pred_xs_ekf = np.empty((sim_steps, x_dim))
pred_xs_ukf = np.empty((sim_steps, x_dim))

for k in tqdm(steps):
    y = plant.step(u)
    true_x = plant.x_true
    pred_x_ekf = ekf.step(y, u)
    pred_x_ukf = ukf.step(y, u)
    true_xs[k, :] = true_x
    pred_xs_ekf[k, :] = pred_x_ekf
    pred_xs_ukf[k, :] = pred_x_ukf

plt.figure(figsize=(12, 9))
plt.plot(steps, true_xs, label='true_x')
plt.plot(steps, pred_xs_ekf, label='pred_x_ekf')
plt.plot(steps, pred_xs_ukf, label='pred_x_ukf')
plt.xlabel(r'$k$')
plt.legend()

plt.show()
